In [ ]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

client = OpenAI()

resp = client.responses.create(
    model="gpt-5-nano-2025-08-07",  # or "gpt-5-nano"
    reasoning={"effort": "low"},
    tools=[{"type": "web_search"}],
    tool_choice="auto",
    # Optional: include the full list of URLs the tool consulted
    include=["web_search_call.action.sources"],
    input="using gbif taxonomy stadnard with web search and your own reaonsing can you detemine what taxa does this text cover: Impact of gun-hunting on diurnal primates in continental Equatorial Guinea	Bushmeat hunting is threatening wildlife populations across west-central Africa, and now poses a greater threat to primates than habitat loss or degradation does in some areas. However, species vary in their abilities to withstand hunting, either because hunters target them differentially or they vary in their vulnerability to a given level of hunting. We studied the impact of current levels of gun-hunting on diurnal primate species in the little-studied Monte Mitra area of Monte Alen National Park, continental Equatorial Guinea. Most bushmeat is currently trapped, but gun-hunting is increasing as shotguns become more available and affordable, allowing targeting of arboreal as well as terrestrial prey. We collected data over 15-mo, via hunter interviews, gun-hunter follows, an offtake survey recording 9374 individuals, and primate surveys covering 408 km of line transects in 2 sites with differing gun-hunting histories. Inside the park, where gun-hunting pressure was recent and light, we found high primate diversity, density, and biomass, with black colobus (Colobus satanas) particularly abundant at 57 individuals/km(2). However, around the village, where gun-hunting was longer-established, though other species such as the guenons still persisted (albeit at lower densities), Colobus satanas were virtually absent. Being slow and large-bodied, Colobus satanas are preferred and susceptible prey, and an early indicator of overhunting. Monte Alen National Park is currently an important stronghold for primates, particularly Colobus satanas, but regulation of the trade and enforcement of hunting bans in the park are urgently needed to safeguard their future and that of other vulnerable species."
)

print(resp.output_text)

# Optional: print the web sources (URLs) returned by the tool call
data = resp.model_dump()  # robust way to inspect the response
for item in data.get("output", []):
    if item.get("type") == "web_search_call":
        sources = item.get("action", {}).get("sources", [])
        if sources:
            print("\nSOURCES:")
            for s in sources:
                print("-", s.get("url"))


In [ ]:
data

In [ ]:
print("You are a biodiversity scientist with strong geography knowledge.\n\nTask:\nGiven ONLY an academic article’s Title and Abstract, extract the study-area geography and return a single JSON object.\n\nFields to return:\n- scope (one of the allowed values)\n- regions (allowed values only)\n- subregions (allowed values only)\n- countries_iso3 (ISO 3166-1 alpha-3 codes, as strings)\n- locales (free-text specific places in the study area; e.g., parks, basins, islands, provinces, rivers, sites)\n- locale_coordinates (lat/lon for locales only if explicitly provided in text OR if you can estimate confidently from general knowledge)\n\nAllowed regions (use EXACT strings only):\n- All Region\n- Americas\n- Asia and the Pacific\n- Africa\n- Europe and Central Asia\n- Antarctica\n\nAllowed subregions (use EXACT strings only):\n- Caribbean\n- South America\n- Mesoamerica\n- North America\n- South Asia\n- Western Asia\n- Oceania\n- South-East Asia\n- North-East Asia\n- Southern Africa\n- Central Africa\n- West Africa\n- East Africa and adjacent islands\n- North Africa\n- Central and Western Europe\n- Eastern Europe\n- Central Asia\n\nRules (be conservative):\n1) Use ONLY geography that refers to the study area / data collection / focal system.\n   - EXCLUDE author affiliations, publisher locations, and incidental mentions unless clearly part of the study area.\n2) If the study is global/broad with no specific geography focus, set:\n   - scope = \"global\"\n   - regions = [\"All Region\"]\n   - subregions = []\n   - countries_iso3 = []\n   - locales = []\n   - locale_coordinates = []\n3) Otherwise, extract the most specific geography supported by text:\n   - If countries are explicitly part of the study area, include their ISO3 in countries_iso3.\n   - If only a subregion/region is stated and countries are not clearly stated, fill regions/subregions and leave countries_iso3 empty.\n4) Locales:\n   - Include place names only if they are part of the study area.\n   - Do NOT guess countries from locales unless the text explicitly links them.\n5) Coordinates (no web search):\n   - If the title/abstract explicitly contains coordinates, include them in locale_coordinates (associate to the nearest locale name if possible; otherwise use the locale string \"unspecified_site\").\n   - If coordinates are not given, you MAY estimate coordinates only when the locale is highly unique and unambiguous (single obvious interpretation).\n   - If there is any ambiguity (same name in multiple countries/regions, unclear which site), do not output coordinates for that locale.\n6) Multi-geo studies: you may return multiple regions/subregions/countries if explicitly supported.\n7) If unsure, return empty arrays rather than guessing.\n\nReasoning style:\n- Think step-by-step internally, but DO NOT reveal your reasoning.\n\nOutput constraints:\n- Output ONLY valid JSON that matches the required schema.\n- Use unique items in arrays.")

In [ ]:
print("G")

In [ ]:
print("Done.")

In [ ]:
print("You are a biodiversity scientist with strong geography knowledge.\n\nTask:\nGiven ONLY an academic article’s Title and Abstract, extract the study-area geography and return a single JSON object.\n\nFields to return:\n- scope (one of the allowed values)\n- regions (allowed values only)\n- subregions (allowed values only)\n- countries_iso3 (ISO 3166-1 alpha-3 codes, as strings)\n- locales (free-text specific places in the study area; e.g., parks, basins, islands, provinces, rivers, sites)\n- locale_coordinates (lat/lon for locales only if explicitly provided in text OR if you can estimate confidently from general knowledge)\n\nAllowed regions (use EXACT strings only):\n- All Region\n- Americas\n- Asia and the Pacific\n- Africa\n- Europe and Central Asia\n- Antarctica\n\nAllowed subregions (use EXACT strings only):\n- Caribbean\n- South America\n- Mesoamerica\n- North America\n- South Asia\n- Western Asia\n- Oceania\n- South-East Asia\n- North-East Asia\n- Southern Africa\n- Central Africa\n- West Africa\n- East Africa and adjacent islands\n- North Africa\n- Central and Western Europe\n- Eastern Europe\n- Central Asia\n\nInterpretation notes (still be conservative):\n- “Study area” means where the data come from, where sampling occurred, where experiments/fieldwork took place, or what geography the focal ecological system covers.\n- It does NOT mean: author addresses, institution locations, journal/publisher locations, funding agency locations, or where the paper was written.\n- If the abstract mentions a place only as background/context (e.g., “similar patterns in Europe”), do not include it unless the study actually analyzes data from there.\n- If the abstract mentions an organism’s native range or an invasion destination AND the paper analyzes that geography, then it counts as study area.\n- When geography is described by biomes/ecoregions (e.g., “Sahel”, “Palearctic”, “Congo Basin”), include the term as a locale if it is the stated study area; add region/subregion only when unambiguous.\n\nGuidance for choosing scope (do not invent new categories):\n- Select the most appropriate single scope value from your allowed list based on the broadness of the study area.\n- Typical patterns:\n  - Global or near-global coverage across continents -> global.\n  - One continent/major region without specific countries -> regional.\n  - One country -> national.\n  - One province/state/island/park/basin within a country -> subnational/local.\n  - Multiple distinct countries (explicitly in the study area) -> the closest “multi-country” equivalent in your allowed scope list.\n\nRegion/subregion mapping reminders (use only if supported by the text):\n- If a country is in Europe, prefer region = Europe and Central Asia; then choose subregion Central and Western Europe vs Eastern Europe if implied.\n- If a country is in Africa, region = Africa; then choose North Africa / West Africa / Central Africa / Southern Africa / East Africa and adjacent islands if implied.\n- If a country is in Asia, region = Asia and the Pacific; then choose South Asia / Western Asia / South-East Asia / North-East Asia / Central Asia if implied.\n- If a country is in the Americas, region = Americas; then choose North America / Mesoamerica / Caribbean / South America if implied.\n- If the text is only at region level (e.g., “Africa”, “Asia-Pacific”, “Americas”), do not force a subregion.\n\nRules (be conservative):\n1) Use ONLY geography that refers to the study area / data collection / focal system.\n   - EXCLUDE author affiliations, publisher locations, and incidental mentions unless clearly part of the study area.\n   - EXCLUDE locations mentioned only in citations, comparisons, or motivation unless the study uses those data.\n\n2) If the study is global/broad with no specific geography focus, set:\n   - scope = \"global\"\n   - regions = [\"All Region\"]\n   - subregions = []\n   - countries_iso3 = []\n   - locales = []\n   - locale_coordinates = []\n   Notes:\n   - “Global meta-analysis”, “worldwide dataset”, “across continents”, “global synthesis” usually implies this case unless the abstract clearly constrains the geography.\n   - If the abstract says “global” but then clearly restricts the dataset to a specific region (e.g., “global review of European rivers”), do NOT use All Region; follow the restricted study area.\n\n3) Otherwise, extract the most specific geography supported by text:\n   - If countries are explicitly part of the study area, include their ISO3 in countries_iso3.\n   - If only a subregion/region is stated and countries are not clearly stated, fill regions/subregions and leave countries_iso3 empty.\n   - Do not infer a country solely from a demonym or adjective unless it is unmistakable (e.g., “Brazilian Amazon” -> Brazil is acceptable).\n   - If a city/locality is named but the country is not stated, do not guess the country unless the locality is globally unambiguous.\n\n4) Countries and ISO3 codes:\n   - Use ISO 3166-1 alpha-3 codes as uppercase strings (e.g., \"BRA\", \"KEN\", \"DEU\").\n   - Include overseas territories or special regions only if they have an ISO3 code AND the text indicates that territory as the study area.\n   - If the abstract uses historic or ambiguous country names (e.g., “Congo”), do not guess which ISO3 unless it is explicitly clarified.\n   - If the abstract lists many countries as possible examples rather than study sites, do not include them.\n\n5) Locales:\n   - Include named geographic entities that are part of the study area (e.g., valleys, districts, cities/towns, villages, protected areas, basins, islands, rivers, lakes, mountain ranges).\n   - Include administrative units explicitly in text (e.g., provinces/states/regions) if they describe the study area.\n   - Include named water bodies/river basins when they define the sampling frame (e.g., “Mekong River basin”, “Lake Victoria”, “Danube”).\n   - Exclude study-internal sampling labels that are not real place names, such as: “Zone A”, “Zone B”, “Zone C”, “Site 1”, “Station 3”, “Plot II”, “Transect 5”, “Locality 22”.\n   - If a zone/site label is the only way the location is described, omit it (leave it out of locales), unless it contains a proper name beyond the label (e.g., “Salkhala locality” is ok; “Zone B” is not).\n   - If the study area is described only by coordinates (no place names), set locales = [\"unspecified_site\"].\n\n6) Coordinates:\n   - If the title/abstract explicitly contains coordinates, include them in locale_coordinates (associate to the nearest locale name if possible; otherwise use the locale string \"unspecified_site\").\n   - Accept common coordinate formats (decimal degrees; degrees/minutes/seconds; N/S/E/W notations). Convert mentally to decimal degrees when feasible, but do not guess missing signs.\n   - If coordinates are not given, you MAY estimate coordinates only when the locale is highly unique and unambiguous (single obvious interpretation).\n   - If there is any ambiguity (same name in multiple countries/regions, unclear which site), do not output coordinates for that locale.\n   - Keep coordinates within valid ranges (lat: -90..90, lon: -180..180).\n   - Preferred representation for each coordinate entry (unless your schema dictates otherwise): an object with keys {\"locale\": <string>, \"latitude\": <number>, \"longitude\": <number>}.\n\n7) Multi-geo studies:\n   - You may return multiple regions/subregions/countries if explicitly supported.\n   - If multiple countries are listed, include them all (deduplicated) only if the abstract indicates they are part of the analyzed study area.\n   - If the abstract says “multiple countries across West Africa” but does not name them, use region/subregion and leave countries_iso3 empty.\n\n8) Granularity and restraint:\n   - Prefer the most specific geographic description supported by the text.\n   - Do not “upgrade” specificity (e.g., don’t infer provinces when only a country is stated).\n   - If unsure, return empty arrays rather than guessing.\n\nOptional self-check (internal only; do not reveal):\n- Did I accidentally include affiliation/publisher geography?\n- Are all region/subregion strings exactly from the allowed lists?\n- Are ISO3 codes plausible and only for explicitly supported countries?\n- Are locales real place names (not sampling labels)?\n- Are coordinates only included when explicit or truly unambiguous?\n\nReasoning style:\n- Think step-by-step internally, but DO NOT reveal your reasoning.\n\nOutput constraints:\n- Output ONLY valid JSON that matches the required schema.\n- Use unique items in arrays (deduplicate).\n- Keep strings exactly as required (case-sensitive for allowed region/subregion values).\n- Do not add extra keys.\n\nJSON skeleton (structure only; do NOT output this skeleton unless it matches the extracted answer):\n{\n  \"scope\": \"\",\n  \"regions\": [],\n  \"subregions\": [],\n  \"countries_iso3\": [],\n  \"locales\": [],\n  \"locale_coordinates\": []\n}\n```")

In [ ]:
print("You are a biodiversity threat assessment expert specializing in the IUCN Threats Classification Scheme.\n\nYou must follow this FIXED multi-stage workflow across separate calls, in this exact order:\n1) STAGE 1: threat_l0  (always first)\n2) STAGE 2: threat_l1  (only when threats_l1_candidates are provided)\n3) STAGE 3: threat_l2  (only when threats_l2_candidates are provided)\n\n────────────────────────────────────────────────────────\nWHAT COUNTS AS A BIODIVERSITY THREAT (this task)\nDirect threats are the proximate human activities or processes that have impacted, are impacting, or may impact\nthe status of the focal taxon/ecosystem/system being studied (e.g., agriculture expansion, logging, fishing,\npollution discharge, invasive species introduction, dams altering flows, etc.).\n\nIMPORTANT: Use ONLY the provided ARTICLE_TEXT (title/abstract/extra extracted text). Do NOT assume threats\nbased on general knowledge of the species, region, or what is “usually” happening there.\n\n────────────────────────────────────────────────────────\nYOU WILL RECEIVE ONLY THESE INPUTS\n- ARTICLE_TEXT: the article title + abstract (+ optionally additional extracted text)\n- (later) threats_l1_candidates: [{\"name\": <label>, \"desc\": <description>}, ...]\n- (later) threats_l2_candidates: [{\"name\": <label>, \"desc\": <description>}, ...]\n\nYou must infer the CURRENT STAGE from which candidate lists are present:\n- If threats_l1_candidates is NOT provided AND threats_l2_candidates is NOT provided → STAGE 1 (threat_l0)\n- Else if threats_l1_candidates IS provided AND threats_l2_candidates is NOT provided → STAGE 2 (threat_l1)\n- Else if threats_l2_candidates IS provided → STAGE 3 (threat_l2)\n\n────────────────────────────────────────────────────────\nABSOLUTE OUTPUT RULES (ALL STAGES)\n- Return ONLY ONE valid JSON object. No extra text. No markdown. No explanations.\n- Output must match EXACTLY the schema for the current stage.\n- Output unique labels only (no duplicates).\n- Never output \"Not Applicable\" together with other labels.\n- Include \"stop_reason\" with value exactly: \"stop\" or \"continue\".\n\nSTOP/CONTINUE RULE (ALL STAGES)\n- If your results are exactly [\"Not Applicable\"] → \"stop_reason\":\"stop\"\n- Otherwise → \"stop_reason\":\"continue\"\n- Exception: STAGE 3 (threat_l2) is final → always \"stop_reason\":\"stop\"\n\n────────────────────────────────────────────────────────\nGENERAL DECISION RULES (ALL STAGES)\nEvidence threshold (title/abstract only):\nA label qualifies if ANY of the following hold:\n(A) The threat is analyzed as an exposure/driver/predictor/treatment or compared across threat gradients.\n(B) The threat is explicitly stated as affecting the focal species/habitat/system (e.g., “impacts of X on Y”,\n    “threatened by…”, “disturbance from …”, “mortality due to …”).\n(C) Review/threat-assessment exception (still conservative): if the paper is clearly a review/synthesis or\n    threat/pressure assessment, label only threats that are explicitly named as key pressures.\n\nTie-breaking (when multiple plausible):\n1) Prefer the most proximate/direct activity/process discussed as causing impact (not an upstream driver).\n2) Prefer the label that matches the measured/treated variable (e.g., “road density”, “nutrient loads”,\n   “bycatch”, “fire frequency”, “dams altering flow”).\n3) If still tied, prefer the label that is most central to the methods/results (not the introduction).\n\nIf there is no clear anthropogenic threat process being studied (e.g., taxonomy, genetics without threats,\npure natural history, methodological paper unrelated to threats) → return [\"Not Applicable\"].\n\n────────────────────────────────────────────────────────\nSTAGE 1 — threat_l0\nGoal: Identify which top-level IUCN threat category(ies) best match the threats evidenced in ARTICLE_TEXT.\n\nWHAT ARE threat_l0?\n- threat_l0 are the top-level IUCN threat classes (broad categories of proximate pressures/activities).\n\nALLOWED threat_l0 LABELS (exact strings only; with definitions/examples):\n- Residential & Commercial Development — Threats from human settlements or other non-agricultural land uses with a substantial footprint  These are threats tied to a defined and relatively compact area, which distinguishes them from those in 4. Transportation & Service Corridors which have a long narrow footprint, and 6. Human Intrusions & Disturbance which do not have an explicit footprint.\n- Agriculture & Aquaculture — Threats from farming and ranching as a result of agricultural expansion and intensification, including silviculture, mariculture and aquaculture (includes the impacts of any fencing around farmed areas)  Threats resulting from the use of agrochemicals, rather than the direct conversion of land to agricultural use, should be included under 9.3 Agricultural & Forestry Effluents. Likewise in cases where conversion to agriculture causes increased runoff and hence sedimentation of rivers and lakes, that is also best treated under 9.3 Agricultural & Forestry Effluents.\n- Energy Production & Mining — Threats from production of non-biological resources  Various forms of water use (for example, dams for hydro power) could also be put in this class, but these threats seemed more related to other threats that involve alterations to hydrologic regimes. As a result, they should go in 7.2 Dams & Water Management/Use\n- Transportation & Service Corridors — Threats from long narrow transport corridors and the vehicles that use them including associated wildlife mortality  This class includes transportation corridors outside of human settlements and industrial developments. These corridors create specific stresses to biodiversity including especially fragmentation of habitats and lead to other threats including farms, invasive species, and poachers.\n- Biological Resource Use — Threats from consumptive use of \\\"wild\\\" biological resources including both deliberate and unintentional harvesting effects; also persecution or control of specific species  Consumptive use means that the resource is removed from the system or destroyed - multiple people cannot use the same resource, as they could under 6. Human Intrusions & Disturbance. Threats in the class can affect both target species (harvest of desired trees or fish species) as well as \\\"collateral damage\\\" to non-target species (trees damaged by felling or fisheries bycatch) and habitats (coral reefs destroyed by trawling). Persecution/control involves harming or killing species because they are considered undesirable. For some of the use threats there is an additional question on whether or not International trade is a significant driver of decline (5.1.1, 5.2.1, 5.3.1, 5.3.2, 5.4.1, 5.4.2).\n- Human Intrusions & Disturbance — Threats from human activities that alter, destroy and disturb habitats and species associated with non-consumptive uses of biological resources  Non-consumptive use means that the resource is not removed - multiple people can use the same resource (for example, birdwatching). These threats typically do not permanently destroy habitat except perhaps in extremely severe manifestations.\n- Natural System Modifications — Threats from actions that convert or degrade habitat in service of “managing” natural or semi-natural systems, often to improve human welfare  This category deals primarily with changes to natural processes such as fire, hydrology, and sedimentation, rather than land use. Thus it does not include threats relating to agriculture (which should be under 2. Agriculture & Aquaculture), or infrastructure (1. Residential & Commercial Development and 4. Transportation & Service Corridors).\n- Invasive & Other Problematic Species, Genes & Diseases — Threats from non-native and native plants, animals, pathogens/microbes, or genetic materials that have or are predicted to have harmful effects on biodiversity following their introduction, spread and/or increase in abundance  After much deliberation it was decided to restrict the use of \\\"invasive species\\\" to refer to non-native species to keep things simple for policy makers. The term \\\"problematic native species\\\" is used instead to refer to native species that have become superabundant or otherwise cause problems. If possible, also record the source of the invasive species and/or conditions that exacerbate their effect. This is the class of threats that covers diseases. Where the Kingdom for a fungal disease is unknown, it should be coded under 8.1.1, 8.2.1 or 8.3.1 and the disease name should be noted in the text field.\n- Pollution — Threats from introduction of exotic and/or excess materials or energy from point and nonpoint sources  This class deals with exotic or excess materials introduced to the environment. There is obviously a fine distinction when the pollution comes from another threat - for example, should an oil spill from a pipeline be classified as 4.2 Utility & Service Lines or 9.2 Industrial & Military Effluents? You will have to exercise some judgement here as to which represents the direct threat in your situation. In some cases, the source of the pollution may be either unknown or from a historical source (e.g., heavy metals buried in sediments). In these cases, you may have to make an educated guess as to which category to assign the pollutant.\n- Geological Events — Threats from catastrophic geological events  Strictly speaking, geological events may be part of natural disturbance regimes in many ecosystems. But they need to be considered a threat if a species or habitat is damaged from other threats and has lost its resilience and is thus vulnerable to the disturbance.\n- Climate Change & Severe Weather —Threats from long-term climatic changes which may be linked to global warming and other severe climatic/weather events that are outside of the natural range of variation, or potentially can wipe out a vulnerable species or habitat  Strictly speaking climatic events may be part of natural disturbance regimes in many ecosystems. But they are a threat if a species or habitat is damaged from other threats and has lost its resilience and is thus vulnerable to the disturbance. Many climatic events may also be increasing in frequency or intensity outside their natural range of variation due to human causes. \n- Other Options — The threats classification scheme is intended to be comprehensive, but as there are often new and emerging threats, this option allows for these new threats to be recorded\n- Not Applicable — Use ONLY if no direct threat process is evidenced as studied/acting in ARTICLE_TEXT.\n\nSTAGE 1 OUTPUT JSON (STRICT):\n{\"results\":[\"<Allowed threat_l0 label>\"],\"stop_reason\":\"<stop|continue>\"}\n(You MAY output multiple labels only if clearly central and distinct.)\n\nSTAGE 1 “MEANING QUESTION” (use internally to decide):\n“What proximate human activity/process (or catastrophic event category) is being studied as impacting the\nfocal biodiversity/system in this paper?”\n\nSTAGE 1 STOP/CONTINUE:\n- If results == [\"Not Applicable\"] → stop_reason=\"stop\"\n- Else → stop_reason=\"continue\"\n\n────────────────────────────────────────────────────────\nSTAGE 2 — threat_l1\nGoal: Choose the best matching level-1 threat label(s) from the provided threats_l1_candidates list.\n\nSTAGE 2 ALLOWED LABELS:\n- You may output ONLY candidate \"name\" values from threats_l1_candidates, OR \"Not Applicable\".\n\nSTAGE 2 OUTPUT JSON (STRICT):\n{\"results\":[\"<candidate name or Not Applicable>\"],\"stop_reason\":\"<stop|continue>\"}\n\nSTAGE 2 decision rules:\n- Match candidates by meaning using their provided \"desc\".\n- Default to ONE candidate.\n- Choose MULTIPLE only if the paper centrally addresses multiple distinct L1 threats.\n- If none of the candidates are clearly evidenced by ARTICLE_TEXT → [\"Not Applicable\"].\n\nSTAGE 2 STOP/CONTINUE:\n- If results == [\"Not Applicable\"] → stop_reason=\"stop\"\n- Else → stop_reason=\"continue\"\n\n────────────────────────────────────────────────────────\nSTAGE 3 — threat_l2 (FINAL)\nGoal: Choose the best matching level-2 threat label(s) from the provided threats_l2_candidates list.\n\nSTAGE 3 ALLOWED LABELS:\n- You may output ONLY candidate \"name\" values from threats_l2_candidates, OR \"Not Applicable\".\n\nSTAGE 3 OUTPUT JSON (STRICT):\n{\"results\":[\"<candidate name or Not Applicable>\"],\"stop_reason\":\"stop\"}\n\nSTAGE 3 decision rules:\n- Match candidates by meaning using their provided \"desc\".\n- Default to ONE candidate.\n- Choose MULTIPLE only if the paper centrally addresses multiple distinct L2 threats.\n- If none of the candidates are clearly evidenced by ARTICLE_TEXT → [\"Not Applicable\"].\n\nSTAGE 3 STOP/CONTINUE:\n- Final stage → stop_reason MUST be \"stop\" (always).",
)

In [110]:
import pandas as pd

In [111]:
df = pd.read_csv("data/gbif/curated/gbif_curated.csv")

/var/folders/sf/ksfgwr354pl2gjdb_f0yn3vr0000gn/T/ipykernel_48482/4040387072.py:1: DtypeWarning: Columns (6,7,8,9,10,13,16,18,19,20,21,22,23,24,25,27,28,29,30,31,32,33,34,35,39,40) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("data/gbif/curated/gbif_curated.csv")


In [112]:
df.head()

,taxonID,datasetID,parentNameUsageID,acceptedNameUsageID,originalNameUsageID,scientificName,scientificNameAuthorship,canonicalName,genericName,specificEpithet,...,marine_curated,freshwater_curated,terrestrial_curated,extinct_curated,hybrid_curated,ageInDays_curated,sizeInMillimeter_curated,massInGram_curated,source,sourceTaxonKey
0,11838827,61a5f178-b5fb-4484-b6d8-9b129739e59d,5.0,NaN,NaN,SH1076374.09FU,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11857967,61a5f178-b5fb-4484-b6d8-9b129739e59d,5.0,NaN,NaN,SH1048761.09FU,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,11693236,61a5f178-b5fb-4484-b6d8-9b129739e59d,5.0,NaN,NaN,SH1317510.09FU,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,11525917,61a5f178-b5fb-4484-b6d8-9b129739e59d,5.0,NaN,NaN,SH1111669.09FU,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,11646423,61a5f178-b5fb-4484-b6d8-9b129739e59d,5.0,NaN,NaN,SH0969972.09FU,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [113]:
df.columns

Index(['taxonID', 'datasetID', 'parentNameUsageID', 'acceptedNameUsageID',
       'originalNameUsageID', 'scientificName', 'scientificNameAuthorship',
       'canonicalName', 'genericName', 'specificEpithet',
       'infraspecificEpithet', 'taxonRank', 'nameAccordingTo',
       'namePublishedIn', 'taxonomicStatus', 'nomenclaturalStatus',
       'taxonRemarks', 'kingdom', 'phylum', 'class', 'order', 'family',
       'genus', 'Unnamed: 23', 'vernaculars_named', 'curated_description',
       'species_id', 'results', 'livingPeriod_curated', 'lifeForm_curated',
       'habitat_curated', 'marine_curated', 'freshwater_curated',
       'terrestrial_curated', 'extinct_curated', 'hybrid_curated',
       'ageInDays_curated', 'sizeInMillimeter_curated', 'massInGram_curated',
       'source', 'sourceTaxonKey'],
      dtype='object')

In [114]:
len(df)

7694321

In [115]:
df[df.taxonomicStatus== 'accepted'].groupby('taxonRank').count()

,taxonID,datasetID,parentNameUsageID,acceptedNameUsageID,originalNameUsageID,scientificName,scientificNameAuthorship,canonicalName,genericName,specificEpithet,...,marine_curated,freshwater_curated,terrestrial_curated,extinct_curated,hybrid_curated,ageInDays_curated,sizeInMillimeter_curated,massInGram_curated,source,sourceTaxonKey
taxonRank,,,,,,,,,,,,,,,,,,,,,
class,747,747,747,0,0,747,0,545,545,0,...,0,0,0,0,0,0,0,0,0,0
family,26624,26624,26624,0,0,26624,0,24711,24711,0,...,0,0,0,0,0,0,0,0,0,0
form,10582,10582,10582,0,551,10582,6942,10582,10582,10582,...,0,0,0,0,0,0,0,0,0,0
genus,266411,266411,266411,0,1,266411,246654,254964,254964,0,...,0,0,0,0,0,0,0,0,0,0
kingdom,8,8,0,0,0,8,0,8,8,0,...,0,0,0,0,0,0,0,0,0,0
order,2787,2787,2787,0,0,2787,0,1974,1974,0,...,0,0,0,0,0,0,0,0,0,0
phylum,257,257,257,0,0,257,0,165,165,0,...,0,0,0,0,0,0,0,0,0,0
species,2598208,2598208,2598208,6,539107,2598208,2517447,2552973,2552973,2550515,...,117264,30142,168945,91538,135,0,0,0,879087,879087
subspecies,192440,192440,192440,1,30238,192440,150993,192330,192330,192330,...,0,0,0,0,0,0,0,0,0,0


In [116]:
df.taxonID.nunique()

7694321

In [117]:
df.taxonRank.unique()

array(['unranked', 'genus', 'kingdom', 'family', 'phylum', 'species',
       'variety', 'subspecies', 'form', 'class', 'order'], dtype=object)

In [118]:
df.taxonomicStatus.unique()

array(['accepted', 'doubtful', 'synonym', 'homotypic synonym',
       'heterotypic synonym', 'proparte synonym', nan], dtype=object)

In [ ]:
path = 'data/gbif/embeddings/batches_meta/batch_0000001_meta.jsonl'

batch_meta = pd.read_json(path, lines=True)

In [ ]:
batch_meta.iloc[0].custom_id

In [ ]:
batch_meta.iloc[0]

In [ ]:
batch_meta.iloc[0].scientific_name

In [ ]:
batch_meta.iloc[0].input_list

In [ ]:
import csv

taxon_path = "data/gbif/extracted/Taxon.tsv"

taxon = pd.read_csv(
    taxon_path,
    sep="\t",
    encoding="utf-8",
    skiprows=1,                 # ignoreHeaderLines="1"
    header=None,                # since meta.xml defines the columns
    quoting=csv.QUOTE_NONE,     # fieldsEnclosedBy="" (no quoting)
    engine="python",            # more tolerant than the C engine
    on_bad_lines="warn",        # or "skip" if you just want it to load
)

In [ ]:
cols = [
    "taxonID",                   # index 0 (the <id index="0" />)
    "datasetID",
    "parentNameUsageID",
    "acceptedNameUsageID",
    "originalNameUsageID",
    "scientificName",
    "scientificNameAuthorship",
    "canonicalName",
    "genericName",
    "specificEpithet",
    "infraspecificEpithet",
    "taxonRank",
    "nameAccordingTo",
    "namePublishedIn",
    "taxonomicStatus",
    "nomenclaturalStatus",
    "taxonRemarks",
    "kingdom",
    "phylum",
    "class",
    "order",
    "family",
    "genus",
]

In [ ]:
taxon.columns = cols

In [ ]:
taxon.columns

In [ ]:
taxonRanks = taxon.taxonRank.unique()

print(taxonRanks)
print(len(taxonRanks))

In [ ]:
taxonStatuses = taxon.taxonomicStatus.unique()
print(taxonStatuses)
print(len(taxonStatuses))

In [ ]:
taxon[taxon.taxonID==2499355]

In [ ]:
taxon[taxon.taxonID==2499354]

In [ ]:
len(taxon)

In [ ]:
taxon[(taxon.taxonomicStatus=='accepted')].groupby('taxonRank').count()

In [ ]:
taxon.taxonID.nunique()

In [ ]:
taxon[taxon.scientificName== 'Tinamus major (J.F.Gmelin, 1789)']

In [ ]:
df.head()

In [ ]:
subset = ['taxonID', 
       'scientificName', 'scientificNameAuthorship',
       'canonicalName', 'genericName', 'specificEpithet',
       'infraspecificEpithet', 'taxonRank', 'taxonomicStatus', 'nomenclaturalStatus',
        'kingdom', 'phylum', 'class', 'order', 'family',
       'genus',  'vernaculars_named', 'curated_description',
       'species_id', 'livingPeriod_curated', 'lifeForm_curated',
       'habitat_curated', 'marine_curated', 'freshwater_curated',
       'terrestrial_curated', 'extinct_curated', 'hybrid_curated',
       'ageInDays_curated', 'sizeInMillimeter_curated', 'massInGram_curated']

In [ ]:
status_sel = ['accepted', 'synonym' 'homotypic synonym' ,'heterotypic synonym'
 'proparte synonym']

rank_sel = ['kingdom', 'phylum', 'class', 'order', 'genus', 'species'  ]


In [ ]:
df[(df.taxonRank.isin(rank_sel)) & ((df.taxonomicStatus.isin(status_sel)))][subset]

In [ ]:
df[(df.taxonRank.isin(rank_sel)) & ((taxon.taxonomicStatus.isin(status_sel)))][subset][:10000].to_csv("data/gbif/curated/gbif_curated_subset.csv", index=False)

In [ ]:
df[(df.taxonRank.isin(rank_sel)) & ((taxon.taxonomicStatus.isin(status_sel)))][subset][:100]

In [ ]:
import re

STOP = {
    "We","I","Our","The","A","An","This","That","These","Those","In","On","At","By","For","From",
    "With","Without","As","And","Or","But","However","Moreover","Therefore","Thus","Results","Methods",
    "Introduction","Discussion","Conclusion","Study","Table","Figure","Data","Analysis"
}

# Optional: expand with journals’ boilerplate words you see often.

def extract_taxon_candidates(text: str, max_terms: int = 80) -> list[str]:
    # 1) High-precision binomials: Genus species (species epithet lowercase, allow hyphen)
    binomials = re.findall(r"\b([A-Z][a-z]{3,})\s+([a-z][a-z-]{1,})\b", text)
    binomial_terms = [f"{g} {s}" for g, s in binomials]

    # 2) Genus + qualifier patterns: "Genus sp.", "Genus spp.", "Genus cf.", etc.
    genus_qual = re.findall(
        r"\b([A-Z][a-z]{3,})\s+(sp\.|spp\.|cf\.|aff\.|nr\.|gen\.|nov\.|subsp\.|var\.)\b",
        text
    )
    genus_terms = [g for g, _ in genus_qual]

    # 3) De-dup, filter stopwords/acronyms
    seen = set()
    out = []
    for t in binomial_terms + genus_terms:
        if t in seen:
            continue
        # filter obvious non-taxa
        head = t.split()[0]
        if head in STOP:
            continue
        if head.isupper():  # acronyms
            continue
        seen.add(t)
        out.append(t)
        if len(out) >= max_terms:
            break

    return out


In [ ]:
abstract = """
Does illegal hunting affect density and behaviour of African grassland birds? A case study on ostrich (Struthio camelus)

Widespread bushmeat hunting represents one of the major threats to many mammals and birds in Africa. We studied the influence of illegal bushmeat hunting on large grassland birds in the Serengeti National Park (SNP) and adjoining protected areas, by using the ostrich (Struthio camelus) as a case study. First, we documented illegal hunting of both small and large birds by using a questionnaire in the villages on the western and eastern side of the SNP. Second, we studied the effect of illegal hunting on density by driving 4,659 km of transects inside SNP and on the adjacent protected areas, where the data were analysed by DISTANCE sampling. Last, we used flight initiation distance (FID, i.e. the distance between an approaching predator (human) and prey when flight is started), to assess possible impacts on behaviour from illegal hunting. We found that people from the western side of the SNP admitted to hunting both small and large grassland birds, and collect ostrich feathers and eggs. Although the Maasai also hunted small birds, only ostrich feathers and eggs of the large grassland birds were used. Surprisingly, we found no significant differences in densities between the SNP and adjoining partially protected areas, but ostriches had longer FID to an approaching human outside the SNP. Currently illegal hunting does not appear to affect the ostrich population, but given the extensive use of birds for consumption more awareness educational programs accompanied by provision of agricultural incentives within the protected areas are needed.
"""
candidates = extract_taxon_candidates(abstract)
print(len(candidates))
print(candidates)

In [ ]:
# pip install taxonerd
# then install one model package (see TaxoNERD README for model tar.gz URLs)  :contentReference[oaicite:1]{index=1}

import spacy

# pick a model:
# - en_ner_eco_md = faster
# - en_core_eco_md_weak = often better recall on common names (per README discussion) :contentReference[oaicite:2]{index=2}
nlp = spacy.load("en_core_eco")  # or "en_ner_eco_md"

def extract_taxon_candidates_taxonerd(text: str, max_terms: int = 200) -> list[str]:
    doc = nlp(text)
    cands = []
    seen = set()
    for ent in doc.ents:
        s = ent.text.strip()
        if not s:
            continue
        key = s.lower()
        if key in seen:
            continue
        seen.add(key)
        cands.append(s)
        if len(cands) >= max_terms:
            break
    return cands


In [ ]:
abstract = """
Does illegal hunting affect density and behaviour of African grassland birds? A case study on ostrich (Struthio camelus)

Widespread bushmeat hunting represents one of the major threats to many mammals and birds in Africa. We studied the influence of illegal bushmeat hunting on large grassland birds in the Serengeti National Park (SNP) and adjoining protected areas, by using the ostrich (Struthio camelus) as a case study. First, we documented illegal hunting of both small and large birds by using a questionnaire in the villages on the western and eastern side of the SNP. Second, we studied the effect of illegal hunting on density by driving 4,659 km of transects inside SNP and on the adjacent protected areas, where the data were analysed by DISTANCE sampling. Last, we used flight initiation distance (FID, i.e. the distance between an approaching predator (human) and prey when flight is started), to assess possible impacts on behaviour from illegal hunting. We found that people from the western side of the SNP admitted to hunting both small and large grassland birds, and collect ostrich feathers and eggs. Although the Maasai also hunted small birds, only ostrich feathers and eggs of the large grassland birds were used. Surprisingly, we found no significant differences in densities between the SNP and adjoining partially protected areas, but ostriches had longer FID to an approaching human outside the SNP. Currently illegal hunting does not appear to affect the ostrich population, but given the extensive use of birds for consumption more awareness educational programs accompanied by provision of agricultural incentives within the protected areas are needed.
"""
candidates = extract_taxon_candidates_taxonerd(abstract)
print(len(candidates))
print(candidates)